<a href="https://colab.research.google.com/github/Harshini006/Deepfake-Detection/blob/master/Deepfake_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os

# Define local clean dataset directory path
CLEAN_DATA_DIR = "dataset_clean"

print("🤖 Ingesting Training Directory...")
train_loader = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_DATA_DIR, 'train'),
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'
)

print("\n🤖 Ingesting Validation Directory...")
val_loader = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_DATA_DIR, 'validation'),
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'
)

# Pull a quick mathematical tensor batch to lock in validation metrics
for images, labels in train_loader.take(1):
    print("\n✅ Preprocessing Pipeline Active!")
    print(f"Tensor Batch Shape: {images.shape}")
    print(f"Labels Batch Shape: {labels.shape}")

🤖 Ingesting Training Directory...
Found 11972 files belonging to 2 classes.

🤖 Ingesting Validation Directory...
Found 7018 files belonging to 2 classes.

✅ Preprocessing Pipeline Active!
Tensor Batch Shape: (32, 224, 224, 3)
Labels Batch Shape: (32, 1)


In [5]:
def create_deepfake_detector():
    model = models.Sequential([
        # Input Spec & Pixel Scaling normalization (0 to 1 range conversion)
        layers.Input(shape=(224, 224, 3)),
        layers.Rescaling(1./255),

        # Convolutional Filter Block 1
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Convolutional Filter Block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Convolutional Filter Block 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Deep Flatten Classifier & Dropout regularization layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),

        # Output Node (Probability logic: Real vs Fake classifier mapping)
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Register and summary tracking
model = create_deepfake_detector()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print("🎯 Model Compilation Complete! GPU Tensor paths locked in.")

🎯 Model Compilation Complete! GPU Tensor paths locked in.


In [7]:
EPOCHS = 10

history = model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=EPOCHS
)

Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 35s 71ms/step - accuracy: 0.8038 - loss: 0.4133 - val_accuracy: 0.8380 - val_loss: 0.3910
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 58ms/step - accuracy: 0.9114 - loss: 0.2173 - val_accuracy: 0.8668 - val_loss: 0.3072
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 23s 62ms/step - accuracy: 0.9485 - loss: 0.1282 - val_accuracy: 0.8585 - val_loss: 0.3962
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 24s 63ms/step - accuracy: 0.9645 - loss: 0.0956 - val_accuracy: 0.8611 - val_loss: 0.4034
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 60ms/step - accuracy: 0.9729 - loss: 0.0715 - val_accuracy: 0.8642 - val_loss: 0.5322
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 29s 77ms/step - accuracy: 0.9817 - loss: 0.0500 - val_accuracy: 0.8537 - val_loss: 0.4583
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 23s 62ms/step - accuracy: 0.9831 - loss: 0.0460 - val_accuracy: 0.8444 - val_loss: 0.4475
Epoch 8/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 40s 61ms/step - accuracy: 0.9871 - loss: 0.0297 - 